# Test-Runner — Backtest-Suite

Führt die `pytest`-Suite unter `tests/` aus (Default: `tests/`, also alles inkl. `tests/backtest/`, `tests/signals/`, `tests/test_data.py`) und wertet das Ergebnis direkt im Notebook aus: Zusammenfassung (bestanden/fehlgeschlagen/übersprungen), pro Testdatei aufgeschlüsselt, sowie die vollen Tracebacks jedes Fehlschlags. Kein separates Notebook pro Modul — nur EIN Runner, der jede beliebige Teilmenge unter `tests/` (via `TEST_TARGET`) ausführen kann.

**Kein Ersatz für `python -m pytest` in der Kommandozeile/CI** — dient als schnelles visuelles Feedback beim Entwickeln in Jupyter, identisches `pytest.ini` (`pythonpath = .`) wird verwendet.

In [28]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import subprocess
import xml.etree.ElementTree as ET
from dataclasses import dataclass

import pandas as pd
from IPython.display import display, HTML

REPO_ROOT = Path("..").resolve()
print(f"Repo-Root: {REPO_ROOT}")

Repo-Root: C:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit


## 1. Konfiguration

`TEST_TARGET` ist relativ zu `REPO_ROOT`, z.B. `"tests"` (Default, alles inkl. `tests/backtest`, `tests/signals`, `tests/test_data.py`), `"tests/backtest"` (nur die Engine-Suite) oder ein einzelnes File wie `"tests/backtest/test_engine.py"`.

In [29]:
TEST_TARGET = "tests"  #!#################### GEÄNDERT: war "tests/backtest" -- lief bisher an tests/test_data.py (neue Kerzenumwandlungs-Tests) und tests/signals vorbei
EXTRA_ARGS: list[str] = []  # z.B. ["-k", "engine"] oder ["-x"] fuer Stop-on-first-failure

## 2. Suite ausführen

Läuft über `subprocess` statt `!pytest`, damit stdout/stderr sauber eingefangen werden UND ein maschinenlesbarer JUnit-XML-Report entsteht (`--junitxml`, in pytest eingebaut -- keine Zusatzabhängigkeit nötig).

In [30]:
REPORT_PATH = REPO_ROOT / "tests" / ".last_run_report.xml"
if REPORT_PATH.exists():
    REPORT_PATH.unlink()  # alter Report soll bei einem abgebrochenen Lauf nicht als "aktuell" missverstanden werden

cmd = [
    sys.executable, "-m", "pytest", TEST_TARGET,
    "--tb=short", "-q", f"--junitxml={REPORT_PATH}", *EXTRA_ARGS,
]
print("$", " ".join(cmd))

proc = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True)
print(proc.stdout[-4000:])
if proc.returncode not in (0, 1):  # 0=alle bestanden, 1=mind. 1 Fehlschlag -- beides "die Suite lief durch"
    print(proc.stderr[-2000:])
print(f"\nExit-Code: {proc.returncode}")

if not REPORT_PATH.exists():
    raise RuntimeError(
        "pytest hat keinen Report erzeugt -- der Lauf ist vermutlich VOR der Testsammlung "
        "gescheitert (z.B. pytest fehlt im Kernel-Environment). stdout/stderr oben pruefen; "
        f"Kernel-Python: {sys.executable}"
    )

$ c:\Users\ilija\.conda\envs\tda-trading\python.exe -m pytest tests --tb=short -q --junitxml=C:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\tests\.last_run_report.xml
........................................................................ [ 40%]
........................................................................ [ 81%]
................................                                         [100%]
============================== warnings summary ===============================
tests/test_data.py::TestResampleOhlcvPanelOnRealData::test_15min_panel_has_correct_shape_and_grid
  c:\Users\ilija\.conda\envs\tda-trading\Lib\site-packages\_pytest\fixtures.py:1313: PytestRemovedIn10Warning: Class-scoped fixture defined as instance method is deprecated.
  Instance attributes set in this fixture will NOT be visible to test methods,
  as each test gets a new instance while the fixture runs only once per class.
  Use @classmethod decorator and set attributes on cls instead.
  S

## 3. Zusammenfassung

In [31]:
@dataclass
class TestCaseResult:
    classname: str
    name: str
    file: str
    time: float
    status: str  # passed|failed|error|skipped
    message: str | None


def parse_junit_report(path: Path) -> tuple[pd.DataFrame, dict]:
    if not path.exists():
        raise FileNotFoundError(f"{path} nicht gefunden -- pytest-Lauf in Abschnitt 2 zuerst ausfuehren.")

    tree = ET.parse(path)
    root = tree.getroot()
    suite = root.find("testsuite") if root.tag == "testsuites" else root

    rows: list[TestCaseResult] = []
    for case in suite.findall("testcase"):
        status = "passed"
        message = None
        for tag, st in (("failure", "failed"), ("error", "error"), ("skipped", "skipped")):
            el = case.find(tag)
            if el is not None:
                status = st
                message = el.get("message") or (el.text or "").strip()
                break
        rows.append(TestCaseResult(
            classname=case.get("classname", ""), name=case.get("name", ""),
            file=case.get("file", case.get("classname", "")),
            time=float(case.get("time", 0.0)), status=status, message=message,
        ))

    df = pd.DataFrame([r.__dict__ for r in rows])
    totals = {
        "tests": int(suite.get("tests", 0)),
        "failures": int(suite.get("failures", 0)),
        "errors": int(suite.get("errors", 0)),
        "skipped": int(suite.get("skipped", 0)),
        "time": float(suite.get("time", 0.0)),
    }
    totals["passed"] = totals["tests"] - totals["failures"] - totals["errors"] - totals["skipped"]
    return df, totals


results_df, totals = parse_junit_report(REPORT_PATH)

status_icon = "PASS" if totals["failures"] == 0 and totals["errors"] == 0 else "FAIL"
print(f"[{status_icon}] {totals['tests']} Tests in {totals['time']:.1f}s")
print(f"  bestanden:      {totals['passed']}")
print(f"  fehlgeschlagen: {totals['failures']}")
print(f"  Fehler:         {totals['errors']}")
print(f"  uebersprungen:  {totals['skipped']}")

[PASS] 176 Tests in 12.5s
  bestanden:      176
  fehlgeschlagen: 0
  Fehler:         0
  uebersprungen:  0


## 4. Pro Testdatei

Aggregiert `results_df` je Datei -- zeigt sofort, welches Modul (`test_engine.py`, `test_tuning.py`, ...) betroffen ist, ohne die volle Liste einzeln durchzusehen.

In [32]:
by_file = (
    results_df
    .assign(file=results_df["file"].apply(lambda p: Path(p).name))
    .groupby("file")
    .agg(tests=("status", "size"),
         passed=("status", lambda s: (s == "passed").sum()),
         failed=("status", lambda s: (s == "failed").sum()),
         error=("status", lambda s: (s == "error").sum()),
         skipped=("status", lambda s: (s == "skipped").sum()),
         time_s=("time", "sum"))
    .sort_values(["failed", "error"], ascending=False)
)
display(by_file)

,tests,passed,failed,error,skipped,time_s
file,,,,,,
tests.backtest.test_cliques.TestActiveAt,3,3,0,0,0,0.048
tests.backtest.test_cliques.TestCliqueTimelineFromCache,7,7,0,0,0,0.285
tests.backtest.test_cliques.TestTimelineCursor,3,3,0,0,0,0.056
tests.backtest.test_config.TestBacktestConfigSerialization,5,5,0,0,0,0.003
tests.backtest.test_config.TestBasketParamsOverride,3,3,0,0,0,0.001
tests.backtest.test_config.TestSplitCalibrationBacktest,5,5,0,0,0,0.002
tests.backtest.test_engine.TestEndToEndConsistency,3,3,0,0,0,3.176
tests.backtest.test_engine.TestNoOpStrategyKeepsEquityFlat,1,1,0,0,0,0.066
tests.backtest.test_engine.TestTimestampGridGuard,1,1,0,0,0,0.018


## 5. Fehlgeschlagene Tests im Detail

Nur relevant, falls Abschnitt 3 Fehlschläge/Fehler meldet -- volle Fehlermeldung je Testfall, direkt aus dem JUnit-Report (identisch zu `--tb=short` in der Konsole).

In [33]:
failed_df = results_df[results_df["status"].isin(["failed", "error"])]

if failed_df.empty:
    print("Keine Fehlschlaege.")
else:
    for _, row in failed_df.iterrows():
        print(f"{'='*80}\n{Path(row['file']).name}::{row['classname'].split('.')[-1]}::{row['name']} [{row['status']}]\n{'-'*80}")
        print(row["message"] or "(keine Meldung im Report)")
        print()

Keine Fehlschlaege.


## 6. Übersprungene Tests

In [34]:
skipped_df = results_df[results_df["status"] == "skipped"]
if skipped_df.empty:
    print("Keine uebersprungenen Tests.")
else:
    display(skipped_df[["file", "classname", "name", "message"]])

Keine uebersprungenen Tests.


## 7. Fazit

_Wird bei jedem Lauf neu bewertet -- kein Freitext zum Pflegen, sondern direkt aus `totals` abgeleitet._

In [35]:
if totals["failures"] == 0 and totals["errors"] == 0:
    display(HTML(f'<div style="padding:10px;background:#1e3a1e;color:#c8f7c5;border-radius:6px;">'
                 f'Alle {totals["tests"]} Tests bestanden ({totals["time"]:.1f}s), '
                 f'{totals["skipped"]} uebersprungen.</div>'))
else:
    display(HTML(f'<div style="padding:10px;background:#3a1e1e;color:#f7c5c5;border-radius:6px;">'
                 f'{totals["failures"] + totals["errors"]} von {totals["tests"]} Tests fehlgeschlagen '
                 f'-- siehe Abschnitt 4/5 fuer Details.</div>'))